# Sera-Z — Latih Model 1 (YOLOv11n-cls) di Colab

Notebook ini sudah final. Jalankan sel dari atas ke bawah, jangan melompat.

- Runtime: **T4 GPU** (`Runtime > Change runtime type > T4 GPU`)
- Repo private: siapkan GitHub token, atau upload `ml.zip`
- Waktu: persiapan data 10-20 menit, training 30 epoch sekitar 1,5-2 jam
- Jangan tutup tab selama training
- Hasil disimpan ke Google Drive di sel kedua dari akhir

In [19]:
!nvidia-smi
import torch
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "tidak ada")

Mon Nov  9 14:56:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             14W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Ambil kode

Kalau folder repo belum ada, sel ini akan clone. Untuk repo private, tempel
GitHub token saat diminta (token tidak ditampilkan). Kalau lebih suka tanpa
token, upload `ml.zip` lalu jalankan sel unzip di bawah.

In [20]:
import os
import subprocess

REPO = "https://github.com/Bisminee/PBL-Mobile-Kelompok2-Semester5.git"
BRANCH = "ml/model1-v2-pipeline"
DIR = "/content/sera-z"

if os.path.isdir(f"{DIR}/ml/scripts"):
    print("Folder repo sudah ada:", DIR)
elif os.path.isdir("/content/PBL-Mobile-Kelompok2-Semester5/ml/scripts"):
    DIR = "/content/PBL-Mobile-Kelompok2-Semester5"
    print("Memakai folder lama:", DIR)
else:
    from getpass import getpass
    token = getpass("GitHub token (kosongkan jika repo publik): ").strip()
    url = REPO.replace("https://", f"https://{token}@") if token else REPO
    subprocess.run(["git", "clone", "--branch", BRANCH, url, DIR], check=True)

os.chdir(DIR)
print("cwd:", os.getcwd())
print("script:", sorted(os.listdir("ml/scripts")))

Folder repo sudah ada: /content/sera-z
cwd: /content/sera-z
script: ['00_download_data.py', '01_prepare_model1.py', '02_balance_train.py', '03_baseline_mobilenetv2.py', '04_train_yolo11n.py', '05_eval_model1.py', '06_export_tflite.py', '07_extract_color_kmeans.py', '08_train_model3_mlp.py', '09_prepare_hf.py', '10_eval_field.py', '11_download_negatives.py', '12_run_model1_pipeline.py']


Alternatif tanpa Git: upload `ml.zip` (dibuat dengan perintah di bawah), lalu
jalankan sel unzip ini.

```powershell
git archive --format=zip -o D:\Temp\ml.zip ml/model1-v2-pipeline ml
```

In [ ]:
# Jalankan hanya jika memakai ml.zip. Lewati kalau clone berhasil.
# !mkdir -p /content/sera-z
# from google.colab import files
# files.upload()
# !unzip -q ml.zip -d /content/sera-z
# import os; os.chdir("/content/sera-z")
# print(os.getcwd(), sorted(os.listdir("ml/scripts")))

## 2. Pasang dependensi

In [21]:
!pip install -q ultralytics datasets pillow pandas scikit-learn matplotlib tqdm
import ultralytics, datasets, torch
print("ultralytics:", ultralytics.__version__)
print("datasets:", datasets.__version__)

ultralytics: 8.4.208
datasets: 4.8.5


## 3. Pastikan script COCO sudah dipatch

Patch ini membuat script membaca kategori COCO di berbagai versi `datasets`.
Aman dijalankan berulang; kalau sudah ada, tidak diubah lagi.

In [22]:
from pathlib import Path

p = Path("ml/scripts/11_download_negatives.py")
kode = p.read_text()

if "kandidat" in kode:
    print("Patch sudah ada.")
else:
    mulai = kode.index("def nama_kategori")
    akhir = kode.index("def main()")
    fungsi_baru = '''def nama_kategori(ds, row) -> set[str] | None:
    # Nama kategori COCO satu baris. None bila tidak bisa dipetakan.
    raw = row.get("objects", {}).get("category", [])
    if not raw:
        return set()
    if all(isinstance(c, str) for c in raw):
        return set(raw)

    fitur = ds.features["objects"]
    kandidat = None
    if hasattr(fitur, "feature"):
        dalam = fitur.feature
        kandidat = dalam.get("category") if isinstance(dalam, dict) else None
    elif isinstance(fitur, dict):
        kandidat = fitur.get("category")
        if kandidat is None and isinstance(fitur.get("feature"), dict):
            kandidat = fitur["feature"].get("category")

    names = getattr(kandidat, "names", None)
    if not names:
        return None
    return {names[c] if isinstance(c, int) else c for c in raw}


'''
    kode = kode[:mulai] + fungsi_baru + kode[akhir:]
    kode = kode.replace(
        "if args.prefer_indoor and not (kategori & KATEGORI_DALAM_RUANG):",
        "if args.prefer_indoor and kategori is not None and not (kategori & KATEGORI_DALAM_RUANG):",
    )
    p.write_text(kode)
    print("Patch diterapkan.")

Patch sudah ada.


## 4. Unduh 800 foto objek bukan pakaian

Kelas `other` berisi laptop, meja, kursi, orang, botol, TV, buku, ruangan.
Dilewati otomatis kalau folder sudah terisi.

In [23]:
from pathlib import Path

jumlah = len(list(Path("ml/data/raw/negatives").rglob("*.jpg")))
print("Foto negatif sekarang:", jumlah)

if jumlah < 700:
    !python ml/scripts/11_download_negatives.py --count 800
else:
    print("Sudah cukup, unduh dilewati.")

Foto negatif sekarang: 800
Sudah cukup, unduh dilewati.


## 5. Siapkan dataset 10 kelas

Perkiraan 10-20 menit. Tanda selesai: `metadata.csv ditulis`.

In [24]:
FORCE = True  # False = pakai dataset yang sudah ada

if FORCE:
    !python ml/scripts/13_prepare_hf.py --force --negatives ml/data/raw/negatives
else:
    print("Dilewati. Set FORCE = True untuk menyiapkan ulang.")

Mengunduh metadata ashraq/fashion-product-images-small (parquet ~270 MB pada run pertama)...
Total baris dataset: 44072
[negatives] 800 foto bukan pakaian ditambahkan sebagai kelas 'other'.

--- Jumlah per split x kelas ---
target  dress  long_pants  other  outerwear  shirt  shoes  shorts  skirt  tshirt
split                                                                           
test       58         213    120        124    470   1277      69     16    1194
train     273         993    560        578   2193   5961     319     73    5572
val        59         213    120        123    470   1278      68     15    1195

[train] 16522 gambar
menyimpan gambar: 100% 16522/16522 [00:13<00:00, 1260.95it/s]
[dedup] 167 gambar identik dibuang.

[val] 3541 gambar
menyimpan gambar: 100% 3541/3541 [00:02<00:00, 1182.28it/s]
[dedup] 89 gambar identik dibuang.

[test] 3541 gambar
menyimpan gambar: 100% 3541/3541 [00:02<00:00, 1375.93it/s]
[dedup] 80 gambar identik dibuang.

metadata.csv ditulis:

## 6. Verifikasi dataset

Harus ada 10 kelas termasuk `other`, dan semua split terisi.

In [25]:
import pandas as pd

m = pd.read_csv("ml/data/processed/model1/metadata.csv")
print(m.groupby(["split", "target"]).size().unstack(fill_value=0).to_string())
print()
print("Kelas:", sorted(m["target"].unique()))
print()
print(m.groupby(["split", "source"]).size().to_string())

target  dress  long_pants  other  outerwear  shirt  shoes  shorts  skirt  tshirt
split                                                                           
test       58         213    120        124    413   1274      64     13    1182
train     272         992    560        576   2067   5957     312     71    5548
val        56         212    120        122    401   1277      65     14    1185

Kelas: ['dress', 'long_pants', 'other', 'outerwear', 'shirt', 'shoes', 'shorts', 'skirt', 'tshirt']

split  source             
test   fashion_products_hf     3341
       negatives                120
train  fashion_products_hf    15795
       negatives                560
val    fashion_products_hf     3332
       negatives                120


## 7. Latih model

30 epoch, batch 64, 224 px. Perkiraan 1,5-2 jam di T4.

In [26]:
!python ml/scripts/04_train_yolo11n.py \
    --data ml/data/processed/model1 \
    --epochs 30 --imgsz 224 --batch 64 \
    --workers 2 --device 0 --name model1_hf_v2

Kelas (9): ['dress', 'long_pants', 'other', 'outerwear', 'shirt', 'shoes', 'shorts', 'skirt', 'tshirt']
Train: 16355 gambar | Val: 3452 gambar

Mulai latih: yolo11n-cls.pt | epochs=30 | imgsz=224

Ultralytics 8.4.208 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=ml/data/processed/model1, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, iou=None, kobj=1.0, line_width=None, lr0=0

## 8. Evaluasi test katalog

In [27]:
!python ml/scripts/05_eval_model1.py \
    --data ml/data/processed/model1 \
    --weights ml/runs/model1_hf_v2/weights/best.pt

import json
d = json.load(open("ml/reports/model1_test_summary.json"))
print("Akurasi:", d["accuracy"], "| Macro F1:", d["macro_f1"])

Ultralytics 8.4.208 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n-cls summary (fused): 47 layers, 1,537,553 parameters, 0 gradients, 3.2 GFLOPs
train: /content/sera-z/ml/data/processed/model1/train... found 16355 images in 9/9 classes ✅ 
val: /content/sera-z/ml/data/processed/model1/val... found 3452 images in 9/9 classes ✅ 
test: /content/sera-z/ml/data/processed/model1/test... found 3461 images in 9/9 classes ✅ 
test: Fast image access ✅ (ping: 0.0±0.0 ms, read: 123.0±66.7 MB/s, size: 2.3 KB)
test: Scanning /content/sera-z/ml/data/processed/model1/test... 3461 images, 0 corrupt: 100% ━━━━━━━━━━━━ 3461/3461 5.1Kit/s 0.7s
test: New cache created: /content/sera-z/ml/data/processed/model1/test.cache
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 217/217 24.5it/s 8.9s
                   all      0.975          1
Speed: 0.1ms preprocess, 1.1ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to /content/sera-z/runs/classify/val
Ult

## 8b. Validasi ambang penolakan

Cari ambang confidence yang menolak objek bukan pakaian tanpa banyak salah menolak pakaian.


In [ ]:
!python ml/scripts/14_threshold_sweep.py \\
    --weights ml/runs/model1_hf_v2/weights/best.pt


## 9. Ekspor TFLite int8

Pastikan output verifikasi tertulis `[1, 10]`.

In [28]:
!pip install -q tensorflow onnx onnxslim onnxruntime

!python ml/scripts/06_export_tflite.py \
    --data ml/data/processed/model1 \
    --weights ml/runs/model1_hf_v2/weights/best.pt \
    --out-name model1_yolo11n_v2.tflite

WARNING ⚠️ 'int8' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ format='tflite' is deprecated as of 8.4.83 and has been replaced by the unified Google LiteRT format. Exporting format='litert' instead. See https://docs.ultralytics.com/integrations/litert
Ultralytics 8.4.208 🚀 Python-3.13.16 torch-2.14.1+cu130 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
YOLO11n-cls summary (fused): 47 layers, 1,537,553 parameters, 0 gradients, 0.4 GFLOPs

PyTorch: starting from 'ml/runs/model1_hf_v2/weights/best.pt' with input shape (1, 3, 224, 224) BCHW and output shape(s) (1, 9) (3.1 MB)
LiteRT: collecting INT8 calibration images from 'data=ml/data/processed/model1'
train: /content/sera-z/ml/data/processed/model1/train... found 16355 images in 9/9 classes ✅ 
val: /content/sera-z/ml/data/processed/model1/val... found 3452 images in 9/9 clas

## 10. Simpan ke Google Drive

Wajib sebelum sesi Colab berakhir.

In [29]:
from google.colab import drive
drive.mount("/content/drive")

!mkdir -p "/content/drive/MyDrive/SeraZ-ML"
!cp -r ml/runs/model1_hf_v2 "/content/drive/MyDrive/SeraZ-ML/"
!cp ml/reports/model1_test_* "/content/drive/MyDrive/SeraZ-ML/"
!cp assets/models/model1_yolo11n_v2.tflite "/content/drive/MyDrive/SeraZ-ML/"

print("Tersimpan di /content/drive/MyDrive/SeraZ-ML")

Mounted at /content/drive
Tersimpan di /content/drive/MyDrive/SeraZ-ML


## 11. Unduh ke laptop

In [31]:
!zip -r -q /content/model1_v2.zip \
    ml/runs/model1_hf_v2 \
    ml/reports \
    assets/models/model1_yolo11n_v2.tflite

from google.colab import files
files.download("/content/model1_v2.zip")
files.download("/content/sera-z/assets/models/model1_yolo11n_v2.tflite")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [33]:
!zip -r -q /content/model1_v2.zip ml/runs/model1_hf_v2 ml/reports assets/models/model1_yolo11n_v2.tflite
from google.colab import files
files.download("/content/model1_v2.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Setelah selesai

1. Unduh `model1_v2.zip` dan `model1_yolo11n_v2.tflite`.
2. Salin `model1_yolo11n_v2.tflite` menimpa `assets/models/yolo11n_cls.tflite`.
3. Pastikan `assets/models/labels.txt` berisi 10 kelas:

```
dress
long_pants
long_sleeve
other
outerwear
shirt
shoes
shorts
skirt
tshirt
```

4. Build ulang aplikasi:

```powershell
flutter clean
flutter pub get
flutter run
```

## Kalau sesi Colab mati

Isi `/content` hilang. Ulangi sel 1 sampai 4, lalu sel 5 dan seterusnya.
Sebelum menutup Colab, pastikan sel 10 (simpan ke Drive) sudah dijalankan.

## Catatan

- Jangan menjalankan training ulang hanya untuk melihat hasil; pakai sel 8.
- Jangan mencampur `labels.txt` 10 kelas dengan model 15 kelas, atau sebaliknya.
- Model hasil notebook ini 10 kelas. Model lama di aplikasi 15 kelas.